In [22]:
from pathlib import Path
import subprocess
import sys

# THAY ĐƯỜNG DẪN NÀY BẰNG REPO NLP CỦA BẠN
NLP_REPO_URL = "https://github.com/Tralaiday/NLP.git"
NLP_PROJECT_ROOT = Path('/content/NLP/Lab7')

if not NLP_PROJECT_ROOT.exists():
    print(f"Đang clone repository từ: {NLP_REPO_URL}...")
    subprocess.run(
        ["git", "clone", NLP_REPO_URL, str(NLP_PROJECT_ROOT)],
        check=True
    )
else:
    print("Repository đã tồn tại. Đang tiến hành cập nhật (git pull)...")
    subprocess.run(
        ["git", "-C", str(NLP_PROJECT_ROOT), "pull", "origin", "main"],
        check=True
    )

# Thêm thư mục dự án vào sys.path để có thể import các module bên trong
if str(NLP_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(NLP_PROJECT_ROOT))

print("Đã kết nối và cập nhật thành công dự án NLP tại:", NLP_PROJECT_ROOT)

Repository đã tồn tại. Đang tiến hành cập nhật (git pull)...
Đã kết nối và cập nhật thành công dự án NLP tại: /content/NLP/Lab7


In [18]:
import math
import torch
import torch.nn.functional as F
from src.multi_head_self_attention import MultiHeadSelfAttention
from src.positional_encoding import PositionalEncoding

# Experiment 1: Positional Encoding

In [19]:
torch.manual_seed(42)

d_model = 4
n_heads = 1

embeddings = {
    "I": torch.tensor([1.0, 0.0, 0.0, 0.0]),
    "love": torch.tensor([0.0, 1.0, 0.0, 0.0]),
    "NLP": torch.tensor([0.0, 0.0, 1.0, 0.0]),
    "dog": torch.tensor([1.0, 1.0, 0.0, 0.0]),
    "bites": torch.tensor([0.0, 1.0, 1.0, 0.0]),
    "man": torch.tensor([0.0, 0.0, 1.0, 1.0])
}

positional_encoding = PositionalEncoding(
    d_model=d_model,
    max_len=10
)

attention_model = MultiHeadSelfAttention(
    d_model=d_model,
    n_heads=n_heads
)

attention_model.eval()
positional_encoding.eval()

def make_input(sentence, use_positional_encoding):
    x = torch.stack([
        embeddings[token] for token in sentence
    ])

    x = x.unsqueeze(0)

    if use_positional_encoding:
        x = positional_encoding(x)

    return x

@torch.no_grad()
def sentence_representation(sentence, use_positional_encoding):
    x = make_input(sentence, use_positional_encoding)
    output, attention = attention_model(x)

    # Mean pooling
    representation = output.mean(dim=1)

    return representation, attention

In [20]:
s1 = ["I", "love", "NLP"]
s2 = ["NLP", "love", "I"]

for use_pe in [False, True]:
    r1, a1 = sentence_representation(s1, use_pe)
    r2, a2 = sentence_representation(s2, use_pe)

    distance = torch.norm(r1 - r2).item()

    print("Positional encoding:", use_pe)
    print("Representation S1:", r1)
    print("Representation S2:", r2)
    print(f"Distance: {distance:.6f}")
    print()

Positional encoding: False
Representation S1: tensor([[-0.2794,  0.4051,  0.1308,  0.4880]])
Representation S2: tensor([[-0.2794,  0.4051,  0.1308,  0.4880]])
Distance: 0.000000

Positional encoding: True
Representation S1: tensor([[-0.3783,  0.3466, -0.1228,  0.6741]])
Representation S2: tensor([[-0.3746,  0.3616, -0.1219,  0.6709]])
Distance: 0.015808



- Dễ thấy khi dùng self-attention và đưa positional information vào thì biểu diễn giữa 2 câu đã có sự khác biệt so với khi không đưa positional information, đó là vì cơ chế này hoạt động như một túi từ (Bag-of-Word)

# Permutation experiment

In [21]:
s1 = ["dog", "bites", "man"]
s2 = ["man", "bites", "dog"]

results = {}

for use_pe in [False, True]:
    r1, _ = sentence_representation(s1, use_pe)
    r2, _ = sentence_representation(s2, use_pe)

    distance = torch.norm(r1 - r2).item()
    results[use_pe] = distance

    print(
        f"PE={use_pe}: "
        f"distance between representations = {distance:.6f}"
    )

PE=False: distance between representations = 0.000000
PE=True: distance between representations = 0.065592


- Không có positional encoding, self-attention không biết token đang ở vị trí nào. Với cùng một tập token và mean pooling, "dog bites man" và "man bites dog" tạo ra representation gần như giống nhau. Khi thêm positional encoding, mỗi vị trí được cộng thêm một vector khác nhau, nên hai câu có representation khác nhau. Vì vậy, positional encoding là thành phần cần thiết để Transformer phân biệt thứ tự token.

# Experiment 2: Number of attention heads